In [2]:
import os
import time
import yaml
from ultralytics import YOLO, RTDETR
import torch
import shutil
import cv2
from ultralytics.models.yolo.classify import ClassificationTrainer

In [ ]:
if not torch.cuda.is_available():
    raise SystemError("CUDA is not accessible. Verify your PyTorch installation includes GPU support.")

print(f"Active Accelerator: {torch.cuda.get_device_name(0)}")
print("Initiating Master Pipeline Execution on local 8GB RTX 4060...\n")

# =====================================================================
# MODEL 1: YOLOv11-Medium (Single-Stage Convolutional Detector)
# =====================================================================
print("--- [PHASE 1] Launching YOLOv11-Medium Training ---")

# Initialize the Medium variant weights natively
model_yolo = YOLO("yolo11m.pt")

# Execution configuration tailored for 8GB VRAM
results_yolo = model_yolo.train(
    data="dataset.yaml",             # Path to your detection dataset config
    epochs=30,                      # Deep training cycle with automatic early stopping patience
    imgsz=640,                       # Standard optimized grid processing resolution
    batch=16,                        # Physical batch size: YOLOv11-m comfortably fits 16 on 8GB VRAM
    device=0,                        # Target dedicated local GPU
    amp=True,                        # Automatic Mixed Precision (FP16) to conserve memory buffer
    
    # --- Targeted Feature & Perspective Augmentations ---
    perspective=0.001,              # Random perspective matrix to simulate steep overhead camera pitch
    scale=0.5,                       # Forceful mosaic downscaling to train multi-distance scale anchors
    degrees=10.0,                    # Gentle rotational variance to handle diagonal entry approaches
    mosaic=1.0,
    mixup=0.15,
    fliplr=0.5,
    
    project="Safety_Pipeline_Runs",
    name="YOLOv11m_Baseline"
)

# Forcefully flush lingering feature map allocations from the GPU VRAM buffer
del model_yolo
torch.cuda.empty_cache()
print("Phase 1 Complete. Hardware memory cache flushed successfully.\n")

Active Accelerator: NVIDIA GeForce RTX 4060 Laptop GPU
Initiating Master Pipeline Execution on local 8GB RTX 4060...

--- [PHASE 1] Launching YOLOv11-Medium Training ---
New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.50 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras

In [5]:
# =====================================================================
# MODEL 2: RT-DETR (Transformer-Based Attention Detector)
# =====================================================================
print("--- [PHASE 2] Launching RT-DETR Training ---")

# We use the natively supported baseline weights. 
model_rtdetr = RTDETR("rtdetr-l.pt")

results_rtdetr = model_rtdetr.train(
    data="dataset.yaml",
    epochs=30,                       
    imgsz=640,
    
    # --- The Ultimate VRAM Safety Protocol ---
    batch=4,                         # Loads only 2 images into VRAM at a time 
    nbs=16,                          # Accumulates gradients across 8 steps to maintain mathematical stability
    
    device=0,
    amp=True,                        # Mixed precision active
    
    # --- Consistent Perspective Augmentations ---
    perspective=0.001,
    scale=0.5,
    degrees=10.0,
    mosaic=1.0,
    mixup=0.15,
    fliplr=0.5,
    
    project="Safety_Pipeline_Runs",
    name="RTDETR_Baseline_Accumulated"
)

# Forcefully clear VRAM allocations
del model_rtdetr
torch.cuda.empty_cache()
print("Phase 2 Complete. Hardware memory cache flushed successfully.\n")

--- [PHASE 2] Launching RT-DETR Training ---
New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.50 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=rtdetr-l.pt, m

/home/grimlock/miniconda3/envs/default/lib/python3.13/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30      4.45G     0.7075      2.255       0.38          6        640: 63% ━━━━━━━╸──── 656/1037 4.8it/s 2:39<1:202


KeyboardInterrupt: 

In [13]:
def generate_classification_dataset(yolo_images_dir, yolo_labels_dir, output_root):
    # The class mapping defined in your dataset.yaml
    classes = {0: "Helmet", 1: "Pagdi", 2: "No-Helmet"}
    
    # Create the output folders automatically
    for class_name in classes.values():
        os.makedirs(os.path.join(output_root, class_name), exist_ok=True)
        
    image_files = [f for f in os.listdir(yolo_images_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    crop_counter = 0
    
    for img_file in image_files:
        img_path = os.path.join(yolo_images_dir, img_file)
        label_path = os.path.join(yolo_labels_dir, img_file.rsplit('.', 1)[0] + '.txt')
        
        if not os.path.exists(label_path):
            continue # Skip images with no labeled heads
            
        # Load the full street frame
        img = cv2.imread(img_path)
        h, w, _ = img.shape
        
        with open(label_path, 'r') as file:
            lines = file.readlines()
            
        for line in lines:
            parts = line.strip().split()
            class_id = int(float((parts[0])))
            
            # YOLO coordinates are normalized (0.0 to 1.0). Convert them back to exact pixels.
            x_center, y_center = float(parts[1]) * w, float(parts[2]) * h
            box_width, box_height = float(parts[3]) * w, float(parts[4]) * h
            
            x1 = int(x_center - (box_width / 2))
            y1 = int(y_center - (box_height / 2))
            x2 = int(x_center + (box_width / 2))
            y2 = int(y_center + (box_height / 2))
            
            # Ensure coordinates don't fall off the edge of the image
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            
            # Slice the image array to create the tight crop
            crop = img[y1:y2, x1:x2]
            
            # Skip invalid, infinitesimally small crops
            if crop.shape[0] == 0 or crop.shape[1] == 0:
                continue
                
            # Save the crop directly into the correct class folder
            class_name = classes[class_id]
            save_path = os.path.join(output_root, class_name, f"crop_{crop_counter}.jpg")
            cv2.imwrite(save_path, crop)
            crop_counter += 1
            
    print(f"Successfully generated {crop_counter} tightly cropped images into {output_root}!")

generate_classification_dataset("Datasets/Detection-Dataset/images/train", "Datasets/Detection-Dataset/labels/train", "Datasets/Head-Crops-Dataset/train")

Successfully generated 11015 tightly cropped images into Datasets/Head-Crops-Dataset/train!


In [7]:
# =====================================================================
# MODEL 3: ResNet-50 (Decoupled Image Classifier Head)
# =====================================================================
print("--- [PHASE 3] Launching ResNet-50 Attribute Classifier ---")

# Loading the native YAML builds the ResNet-50 architecture using Ultralytics layers,
# completely bypassing the PyTorch dictionary crash while preserving the architecture.
model_resnet = YOLO("yolov8-cls-resnet50.yaml") 

results_resnet = model_resnet.train(
    data="/home/grimlock/Downloads/Project/Models/Datasets/Head-Crops-Dataset",       
    epochs=30,
    patience=10,
    imgsz=224,                       
    batch=64,                        
    device=0,
    amp=True,                        
    
    # Custom pixel-level noise for cropped patches
    degrees=5.0,
    fliplr=0.5,
    
    project="Safety_Pipeline_Runs",
    name="ResNet50_Classifier"
)

# Forcefully flush the VRAM buffer
del model_resnet
torch.cuda.empty_cache()
print("Phase 3 Complete. Hardware memory cache flushed successfully.\n")

--- [PHASE 3] Launching ResNet-50 Attribute Classifier ---
WARNING ⚠️ no model scale passed. Assuming scale='n'.
YOLOv8-cls-resnet50 summary: 145 layers, 27,413,032 parameters, 27,413,032 gradients, 70.6 GFLOPs
New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.50 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/grimlock/Downloads/Project/Models/Datasets/Head-Crops-Dataset, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fr

Exception in thread Thread-74 (_pin_memory_loop):
Traceback (most recent call last):
  File "/home/grimlock/miniconda3/envs/default/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/home/grimlock/miniconda3/envs/default/lib/python3.13/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/home/grimlock/miniconda3/envs/default/lib/python3.13/threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/grimlock/miniconda3/envs/default/lib/python3.13/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
    ~~~~~~~~~~~^^
  File "/home/grimlock/miniconda3/envs/default/lib/python3.13/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
  File "/home/grimlock/min

KeyboardInterrupt: 

In [ ]:
def secure_and_export_models():
    print("--- Initiating Weight Verification and Export Pipeline ---")
    
    # Define centralized production storage
    deploy_dir = "./Final_Deployment_Weights"
    os.makedirs(deploy_dir, exist_ok=True)
    
    # Map run directories to target output filenames
    models_to_process = {
        "YOLOv11m": "./Safety_Pipeline_Runs/YOLOv11m_Baseline/weights/best.pt",
        "RTDETR_R18": "./Safety_Pipeline_Runs/RTDETR_R18_Accumulated/weights/best.pt",
        "ResNet50": "./Safety_Pipeline_Runs/ResNet50_Classifier/weights/best.pt"
    }
    
    for model_name, weight_path in models_to_process.items():
        if not os.path.exists(weight_path):
            print(f"[WARNING] Original weights not found for {model_name} at {weight_path}. Skipping.")
            continue
            
        print(f"\nProcessing {model_name}...")
        
        # 1. Copy pure PyTorch weights to deployment folder
        pt_dest = os.path.join(deploy_dir, f"{model_name}_best.pt")
        shutil.copy(weight_path, pt_dest)
        print(f"Secured PyTorch weights: {pt_dest}")
        
        # 2. Load model into native framework interface
        # ResNet utilizes the standard YOLO wrapper class inside Ultralytics
        model = RTDETR(pt_dest) if "RTDETR" in model_name else YOLO(pt_dest)
        
        # 3. Export to ONNX for accelerated CPU/GPU edge inference
        # imgsz enforces dynamic or static input bounding profiles
        export_size = 224 if "ResNet" in model_name else 640
        onnx_file = model.export(
            format="onnx", 
            imgsz=export_size, 
            half=True,       # FP16 precision export to match local RTX 4060 inference limits
            simplify=True    # Simplifies computational graphs for downstream edge runtimes
        )
        print(f"Successfully exported optimized ONNX graph: {onnx_file}")

if __name__ == "__main__":
    secure_and_export_models()
    print("\n=======================================================")
    print(f"All deployment assets successfully staged in: ./Final_Deployment_Weights/")
    print("=======================================================")